In [1]:
from dataclasses import dataclass
from typing import Optional
from yargy import Parser, rule, or_, and_, not_
from yargy.predicates import gram, dictionary, type, eq, normalized, gte, lte, is_title, caseless, is_capitalized, Predicate
from yargy.interpretation import fact, attribute
from yargy.relations import gnc_relation
import gzip

In [2]:
@dataclass
class Entry:
    name: str
    birth_date: Optional[str] = None
    birth_place: Optional[str] = None

    def to_dict(self):
        return {
            "name": self.name,
            "birth_date": self.birth_date,
            "birth_place": self.birth_place,
        }

In [3]:
PersonName = fact("PersonName", ["first", "last", "middle"])
Date = fact("Date", ["day", "month", "year"])
Place = fact("Place", ["value"])
EntryFact = fact("EntryFact", ["person", "date", "place"])

In [4]:
class LenPredicate(Predicate):
    def __init__(self, min_len):
        self.min_len = min_len
        super().__init__()

    def __call__(self, token):
        return len(token.value) >= self.min_len

class NotInSetPredicate(Predicate):
    def __init__(self, stop_words):
        self.stop_words = set(w.lower() for w in stop_words)
        super().__init__()

    def __call__(self, token):
        return token.value.lower() not in self.stop_words

In [5]:
MIN_NAME_LEN = LenPredicate(2)
MIN_PATRONYMIC_LEN = LenPredicate(3)
MIN_SURNAME_LEN = LenPredicate(2)

COMMON_PREPS = {"в", "из", "на", "под", "над", "с", "к", "за", "у", "по", "от", "до", "о"}
NOT_PREP = NotInSetPredicate(COMMON_PREPS)

def prop_name():
    return and_(
        gram("Name"),
        NOT_PREP,
        MIN_NAME_LEN,
        is_capitalized()
    )

def prop_surname():
    return and_(
        gram("Surn"),
        NOT_PREP,
        MIN_SURNAME_LEN,
        is_capitalized()
    )

def prop_patronymic():
    return and_(
        gram("Patr"),
        NOT_PREP,
        MIN_PATRONYMIC_LEN,
        is_capitalized()
    )

In [6]:
YEAR_SUFFIX = dictionary({"год", "года", "году", "г", "г.", "р.", "г.р.", "г.р", "р", "рожд.", "рождения", "года рождения"})

MONTHS = {
    "январь": 1,
    "января": 1,
    "февраль": 2,
    "февраля": 2,
    "март": 3,
    "марта": 3,
    "апрель": 4,
    "апреля": 4,
    "май": 5,
    "мая": 5,
    "июнь": 6,
    "июня": 6,
    "июль": 7,
    "июля": 7,
    "август": 8,
    "августа": 8,
    "сентябрь": 9,
    "сентября": 9,
    "октябрь": 10,
    "октября": 10,
    "ноябрь": 11,
    "ноября": 11,
    "декабрь": 12,
    "декабря": 12,
}

NAME_PART = or_(
    prop_name(),
    prop_surname(),
    prop_patronymic()
)

NAME = or_(
    #ФИО
    rule(
        prop_surname().interpretation(PersonName.last),
        prop_name().interpretation(PersonName.first),
        prop_patronymic().interpretation(PersonName.middle).optional(),
    ),
    #ФОИ
    rule(
        prop_name().interpretation(PersonName.first),
        prop_patronymic().interpretation(PersonName.middle).optional(),
        prop_surname().interpretation(PersonName.last),
    ),
    #ИФ
    rule(
        prop_name().interpretation(PersonName.first),
        prop_surname().interpretation(PersonName.last).optional(),
    ),
    #ИО
    rule(
        prop_name().interpretation(PersonName.first),
        prop_patronymic().interpretation(PersonName.middle).optional(),
    ),
).interpretation(PersonName)


MONTH_WORD = dictionary(MONTHS).interpretation(Date.month.custom(MONTHS.get))

DAY = and_(type("INT"), gte(1), lte(31)).interpretation(Date.day.custom(int))
YEAR = and_(type("INT"), gte(0), lte(2026)).interpretation(Date.year.custom(int))


FULL_DATE = rule(DAY.optional(), MONTH_WORD.optional(), YEAR, YEAR_SUFFIX.optional())

DOTTED_DATE = rule(
    type("INT").interpretation(Date.day.custom(int)),
    eq("."),
    type("INT").interpretation(Date.month.custom(int)),
    eq("."),
    type("INT").interpretation(Date.year.custom(int)),
).interpretation(Date)

DATE = or_(FULL_DATE.interpretation(Date), DOTTED_DATE)

# Место рождения
PLACE_PREP = or_(normalized("в"), normalized("из"), normalized("город"), normalized("г."))

# # вариант типа: Санкт-Петербург
CAPITALIZED_GEO = rule(
    and_(
        is_capitalized(),
        not_(gram("Name")),
        not_(gram("Surn")),
        not_(gram("Patr")),
        not_(type("INT")),
    ),
    eq('-').optional(),
    and_(
        is_capitalized(),
        not_(gram("Name")),
        not_(gram("Surn")),
        not_(gram("Patr")),
        not_(type("INT")),
    ).optional()
    )


PLACE = rule(PLACE_PREP, CAPITALIZED_GEO.interpretation(Place.value)).interpretation(
    Place
)


# контекст у дня рождения
BIRTH_PHRASE = or_(
    rule(normalized("родиться")),
    rule(normalized("родился")),
    rule(normalized("родилась")),
    rule(normalized("уроженец")),
    rule(normalized("уроженка")),
    rule(dictionary({"дата", "день"}), dictionary({"рождения", "рождение"})),
    rule(normalized("дата"), dictionary({"рождения", "рождение"})),
)

BIRTH_EVENT = rule(
    NAME.interpretation(EntryFact.person),
    BIRTH_PHRASE,
    DATE.interpretation(EntryFact.date).optional(),
    PLACE.interpretation(EntryFact.place).optional(),
)

# Вариант: "ФИО, 12.08.1978, г.р., в Москве"
BIRTH_ABBR = rule(
    NAME.interpretation(EntryFact.person),
    DATE.interpretation(EntryFact.date),
    YEAR_SUFFIX.optional(),
    PLACE.interpretation(EntryFact.place).optional(),
)


BIRTH_CONTEXT = or_(
    rule(normalized("родиться")),
    rule(normalized("родился")),
    rule(normalized("родилась")),
    rule(normalized("родился"), normalized("в")),
    rule(normalized("родилась"), normalized("в")),
    rule(dictionary({"дата", "день"}), normalized("рождения")),
    rule(caseless("г."), caseless("р.")),
    rule(caseless("д."), caseless("р.")),
)

# имя + контекст + дата + место
RULE_1 = rule(
    NAME.interpretation(EntryFact.person),
    BIRTH_CONTEXT,
    DATE.interpretation(EntryFact.date).optional(),
    normalized("в").optional(),
    PLACE.interpretation(EntryFact.place).optional(),
)

# имя + контекст + место + дата
RULE_2 = rule(
    NAME.interpretation(EntryFact.person),
    BIRTH_CONTEXT,
    normalized("в").optional(),
    PLACE.interpretation(EntryFact.place).optional(),
    DATE.interpretation(EntryFact.date).optional()
)

# имя + дата + (контекст) + место
RULE_3 = rule(
    NAME.interpretation(EntryFact.person),
    normalized("в").optional(),
    DATE.interpretation(EntryFact.date).optional(),
    BIRTH_CONTEXT.optional(),
    PLACE.interpretation(EntryFact.place).optional(),
)

# контекст + дата + место + имя
RULE_4 = rule(
    BIRTH_CONTEXT,
    DATE.interpretation(EntryFact.date).optional(),
    normalized("в").optional(),
    PLACE.interpretation(EntryFact.place).optional(),
    NAME.interpretation(EntryFact.person),
)

# контекст + место + имя + дата
RULE_5 = rule(
    BIRTH_CONTEXT,
    PLACE.interpretation(EntryFact.place).optional(),
    NAME.interpretation(EntryFact.person),
    DATE.interpretation(EntryFact.date).optional()
)

# имя + место + контекст + дата
RULE_6 = rule(
    NAME.interpretation(EntryFact.person),
    normalized("в").optional(),
    PLACE.interpretation(EntryFact.place).optional(),
    BIRTH_CONTEXT.optional(),
    DATE.interpretation(EntryFact.date).optional(),
)

# имя и дата
RULE_7 = rule(
    NAME.interpretation(EntryFact.person),
    eq(',').optional(),
    DATE.interpretation(EntryFact.date),
    YEAR_SUFFIX.optional(),
)

# имя + контекст + место + дата
RULE_8 = rule(
    NAME.interpretation(EntryFact.person),
    BIRTH_CONTEXT,
    PLACE.interpretation(EntryFact.place).optional(),
    DATE.interpretation(EntryFact.date).optional()
)

ENTRY_RULE = or_(
    RULE_1,
    RULE_2,
    RULE_3,
    RULE_4,
    RULE_5,
    RULE_6,
    RULE_7,
    RULE_8
).interpretation(EntryFact)

In [7]:
def format_name(person):
    if not person:
        return None
    parts = []
    if getattr(person, "last", None):
        parts.append(person.last)
    if getattr(person, "first", None):
        parts.append(person.first)
    if getattr(person, "middle", None):
        parts.append(person.middle)
    return " ".join(parts) if parts else None


def format_date(date):
    if not date:
        return None
    try:
        if hasattr(date, "day") and hasattr(date, "month") and hasattr(date, "year"):
            months = [
                "",
                "января",
                "февраля",
                "марта",
                "апреля",
                "мая",
                "июня",
                "июля",
                "августа",
                "сентября",
                "октября",
                "ноября",
                "декабря",
            ]
            return f"{date.day} {months[date.month]} {date.year}"
        elif hasattr(date, "year"):
            return str(date.year)
    except (TypeError, IndexError, AttributeError):
        pass
    return None

def format_place(place):
    if place:
        return place.value
    return None


parser = Parser(ENTRY_RULE)

def extract_entries(text: str) -> list[Entry]:
    matches = parser.findall(text)
    results = []
    for match in matches:
        f = match.fact
        if not f or not f.person:
            continue
        name = format_name(f.person)
        birth_date = format_date(f.date)
        birth_place = format_place(f.place)
        results.append(Entry(name=name, birth_date=birth_date, birth_place=birth_place))
    return results

In [8]:
first_sentence2wath = 15
i = 0
entries = []
with gzip.open("news.txt.gz", "rt", encoding="utf-8") as f:
    for line in f:
        *_, text = line.strip().split('\t')
        entries.extend(temp:=extract_entries(text))
        if i < first_sentence2wath:
            print(temp)
        i += 1

[Entry(name='Джиралья', birth_date=None, birth_place=None)]
[Entry(name='Матс Сундин', birth_date=None, birth_place=None), Entry(name='Мортса', birth_date=None, birth_place=None), Entry(name='Сундин', birth_date=None, birth_place=None), Entry(name='Мортса', birth_date=None, birth_place=None), Entry(name='Сундин', birth_date=None, birth_place=None), Entry(name='Сундина', birth_date=None, birth_place=None), Entry(name='Сундин', birth_date=None, birth_place=None), Entry(name='Олимпиаде', birth_date=None, birth_place=None), Entry(name='Сундин', birth_date=None, birth_place=None), Entry(name='Юргордене', birth_date=None, birth_place=None), Entry(name='Сундина', birth_date=None, birth_place=None), Entry(name='Сундин', birth_date=None, birth_place=None)]
[Entry(name='Филиппов Владимир', birth_date=None, birth_place=None), Entry(name='Фахриеву Ильгизу', birth_date=None, birth_place=None), Entry(name='Прохоровым Михаилом', birth_date=None, birth_place=None), Entry(name='АКАР', birth_date=None, 

In [ ]:
import json

entries_dict = [e.to_dict() for e in entries]
with open("entries.json", "w", encoding="utf-8") as f:
    json.dump(entries_dict, f, ensure_ascii=False, indent=2)

for e in entries:
    if e.birth_date or e.birth_place:
        print(e)